# Publish LineSegments to Evo from Shapefiles

This notebook demonstrates how to convert shapefiles to LineSegments objects and publish them to your Evo workspace.

## What This Notebook Does

1. **Authentication**: Connects to Evo using ServiceManagerWidget
2. **Setup**: Installs dependencies and imports the shapefile converter
3. **Publishing**: Converts shapefiles to LineSegments format and publishes multiple examples:
   - Natural Earth rivers (PolyLine geometry)
   - Geographic lines (PolyLine geometry)
   - Land boundaries (Polygon converted to LineSegments with CRS override)
   - Populated places (Pointset examples)
4. **Verification**: Shows how to download and inspect published objects

## LineSegments Schema

The converter produces LineSegments objects with:
- **Segments**: Vertices and indices defining line/polygon geometry
- **Parts**: Chunks with offsets/counts, plus attributes from the shapefile DBF
- **Bounding Box**: Spatial extent calculated from all vertices
- **CRS**: Coordinate reference system (from .prj file or can be overridden)

In [ ]:
from evo.notebooks import ServiceManagerWidget

# Replace "your-client-id" with your Evo app client ID
manager = await ServiceManagerWidget.with_auth_code(client_id="your-client-id").login()

## Section 2: Import Required Libraries

Install and import the shapefile converter and supporting modules.

In [ ]:
%pip install pyshp --quiet

print("pyshp is installed")

In [ ]:
import sys
from importlib import import_module
from pathlib import Path

# Find converter source tree by searching up from current working directory
# Looks for: packages/shp/src and packages/common/src
cwd = Path.cwd()
workspace_candidates = [cwd] + list(cwd.parents)

repo_root = next(
    (
        p
        for p in workspace_candidates
        if (p / "packages" / "shp" / "src" / "evo" / "data_converters" / "shp" / "importer" / "shp_to_evo.py").exists()
        and (p / "packages" / "common" / "src" / "evo" / "data_converters" / "common" / "__init__.py").exists()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError(
        f"Could not locate converter source tree (packages/shp and packages/common) from {cwd}. "
        "Please run this notebook from within the geosoft-converters workspace."
    )

shp_src = str(repo_root / "packages" / "shp" / "src")
common_src = str(repo_root / "packages" / "common" / "src")

# Remove prior converter source entries to avoid mixing repositories.
sys.path = [
    p
    for p in sys.path
    if "\\geosoft-converters\\packages\\shp\\src" not in p.lower()
    and "\\geosoft-converters\\packages\\common\\src" not in p.lower()
    and "\\evo-data-converters\\packages\\shp\\src" not in p.lower()
    and "\\evo-data-converters\\packages\\common\\src" not in p.lower()
]

sys.path.insert(0, shp_src)
sys.path.insert(0, common_src)

# Ensure installed evo package can discover local evo subpackages.
evo = import_module("evo")
for evo_subpkg_path in [
    str(Path(common_src) / "evo"),
    str(Path(shp_src) / "evo"),
]:
    if evo_subpkg_path not in evo.__path__:
        evo.__path__.insert(0, evo_subpkg_path)

# Clear cached converter modules so imports resolve from selected source root.
for name in list(sys.modules):
    if name.startswith("evo.data_converters.common") or name.startswith("evo.data_converters.shp"):
        sys.modules.pop(name, None)

convert_shp = import_module("evo.data_converters.shp.importer.shp_to_evo").convert_shp

print("✅ All required libraries imported successfully")
print(f"Converter source root: {repo_root}")

## Section 3: Publish LineSegments Example

Convert and publish a shapefile as LineSegments objects to your workspace. 

The pattern below works for any shapefile type (rivers, polygons, points, etc.). Update the `<YOUR_DATA_PATH>` placeholder with the actual path to your shapefile data.

In [ ]:
# Publish shapefiles as LineSegments, Pointsets, or other Evo objects
# The converter automatically detects geometry type and creates the appropriate schema:
#   - Point/MultiPoint shapefiles → Pointset objects
#   - Line/MultiLine shapefiles → LineSegments objects
#   - Polygon/MultiPolygon shapefiles → LineSegments objects (converted from boundaries)

from pathlib import Path

# ============================================================================
# TODO: Configure these for your data
# ============================================================================
shapefile_name = "ne_50m_rivers_lake_centerlines"  # Change to your shapefile name
data_directory = Path(r"<YOUR_DATA_PATH>")  # Update to your data location
upload_path = "natural_earth/50m/rivers"  # Upload path in Evo workspace
tags = {"Source": "Natural Earth 1:50m", "Dataset": "rivers_lake_centerlines", "Scale": "50m"}
crs_override = None  # Set to {"epsg_code": 4326} to override CRS if needed
# ============================================================================

# Locate and validate shapefile components
shp_path = data_directory / f"{shapefile_name}.shp"
shx_path = data_directory / f"{shapefile_name}.shx"
dbf_path = data_directory / f"{shapefile_name}.dbf"
prj_path = data_directory / f"{shapefile_name}.prj"
cpg_path = data_directory / f"{shapefile_name}.cpg"

required = [shp_path, shx_path, dbf_path]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required shapefile parts: {missing}")

print(f"Publishing {shapefile_name}...")
print(f"Source: {shp_path}")

# Convert and publish
published_objects = convert_shp(
    filepath=str(shp_path),
    filepath_shx=str(shx_path),
    filepath_dbf=str(dbf_path),
    filepath_prj=str(prj_path) if prj_path.exists() else None,
    filepath_cpg=str(cpg_path) if cpg_path.exists() else None,
    upload_path=upload_path,
    service_manager_widget=manager,
    publish_objects=True,
    overwrite_existing_objects=True,
    crs_override=crs_override,
    tags=tags,
)

print(f"\n✅ Published {len(published_objects)} object(s):")
for i, metadata in enumerate(published_objects, start=1):
    print(f"{i}. {metadata.name}")
    print(f"   ID: {metadata.id}")
    print(f"   Path: {metadata.path}")